EDA y Feature Engineering — Calificación inicial de demandas de alimentos
Objetivo: describir los 101 expedientes, verificar la selección de 14 casos aptos, evaluar calidad y extensión de los textos y documentar características textuales sin usar el auto judicial como predictor. Este cuaderno no modifica las fuentes.

Alcance: fijación inicial de alimentos en proceso único. Las etiquetas proceden de validación experta. La evaluación de modelos debe usar únicamente textos de demanda como entrada.


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

# Ejecutar desde la raíz D:\CalificacionDemanda o desde notebooks/.
ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "dataset_filtrado_101.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "processed"
OUT = RESULTS / "eda"
OUT.mkdir(parents=True, exist_ok=True)

ruta = DATA / "dataset_filtrado_101.csv"
assert ruta.exists(), f"No se encontró: {ruta}"
df = pd.read_csv(ruta, dtype=str, keep_default_na=False)
print("Proyecto:", ROOT)
print("Expedientes:", len(df), "| Columnas:", len(df.columns))
display(pd.DataFrame({"columna": df.columns, "no_vacios": [(df[c].str.strip() != "").sum() for c in df.columns]}))

AssertionError: No se encontró: d:\CalificacionDemanda\results\data\processed\dataset_filtrado_101.csv

#integridad y alcance

In [ ]:
requeridas = {"caso_id", "dentro_alcance", "apto_entrenamiento", "etiqueta_inicial", "texto"}
assert requeridas.issubset(df.columns), f"Faltan columnas: {requeridas - set(df.columns)}"
assert df["caso_id"].is_unique, "Existen caso_id duplicados; revisar antes de continuar"
df["apto_bool"] = df["apto_entrenamiento"].str.strip().str.lower().isin(["true", "1", "si", "sí"])
df["n_caracteres"] = df["texto"].str.len()
df["n_palabras"] = df["texto"].str.split().str.len().fillna(0).astype(int)
print("ALCANCE")
display(df["dentro_alcance"].value_counts(dropna=False).rename_axis("estado").reset_index(name="casos"))
print("APTOS:", int(df["apto_bool"].sum()))
print("ETIQUETAS APTAS")
display(df.loc[df.apto_bool, "etiqueta_inicial"].value_counts().rename_axis("etiqueta").reset_index(name="casos"))
assert (~df.loc[df.apto_bool, "dentro_alcance"].eq("SI")).sum() == 0, "Hay aptos fuera de alcance"
assert (df.loc[df.apto_bool, "n_caracteres"] > 0).all(), "Hay aptos sin texto"
print("Validaciones de integridad básicas: OK")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df["dentro_alcance"].value_counts().plot.bar(ax=axes[0], title="Estado del alcance", rot=0)
df.loc[df.apto_bool, "etiqueta_inicial"].value_counts().plot.bar(ax=axes[1], title="Etiquetas aptas", rot=0)
axes[0].set_ylabel("Expedientes")
axes[1].set_ylabel("Expedientes")
plt.tight_layout()
plt.savefig(OUT / "01_alcance_y_etiquetas.png", dpi=160, bbox_inches="tight")
plt.show()

2. Extensión de textos y calidad de extracción

In [ ]:
print("Longitud de los textos (todos los casos)")
display(df[["n_caracteres", "n_palabras"]].describe().round(1))
print("Longitud de los textos (aptos)")
display(df.loc[df.apto_bool, ["n_caracteres", "n_palabras"]].describe().round(1))
fig, ax = plt.subplots(figsize=(9, 4))
for label, subset in [("Aptos", df[df.apto_bool]), ("No aptos", df[~df.apto_bool])]:
    ax.hist(subset["n_palabras"], bins=20, alpha=0.55, label=label)
ax.set(title="Distribución de longitud de textos", xlabel="Palabras", ylabel="Expedientes")
ax.legend()
plt.tight_layout()
plt.savefig(OUT / "02_longitud_textos.png", dpi=160, bbox_inches="tight")
plt.show()

ocr_path = DATA / "textos_demandas_ocr.csv"
if ocr_path.exists():
    ocr = pd.read_csv(ocr_path, dtype=str, keep_default_na=False)
    print("Registros OCR:", len(ocr))
    for campo in ["estado", "errores_ocr", "paginas_ocr"]:
        if campo in ocr:
            print("Campo:", campo)
            if campo == "estado":
                display(ocr[campo].value_counts().rename_axis("estado").reset_index(name="registros"))
            else:
                display(pd.to_numeric(ocr[campo], errors="coerce").describe().to_frame(name=campo).round(1))
else:
    print("No se encontró textos_demandas_ocr.csv; se omite el análisis específico de OCR")

3. Feature Engineering (solo demandas)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
aptos = df.loc[df.apto_bool].copy()
aptos["longitud_media_palabra"] = aptos["texto"].apply(lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0)
display(aptos.groupby("etiqueta_inicial")[["n_caracteres", "n_palabras", "longitud_media_palabra"]].agg(["count", "median", "mean"]).round(2))

if len(aptos) and aptos["texto"].str.strip().ne("").any():
    v = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b\w\w+\b", max_features=25)
    matriz = v.fit_transform(aptos["texto"])
    frecuencias = pd.Series(np.asarray(matriz.sum(axis=0)).ravel(), index=v.get_feature_names_out()).sort_values(ascending=False)
    display(frecuencias.head(15).rename_axis("termino").reset_index(name="frecuencia"))
    fig, ax = plt.subplots(figsize=(9, 5))
    frecuencias.head(15).sort_values().plot.barh(ax=ax, title="Términos frecuentes (aptos)")
    ax.set_xlabel("Frecuencia total")
    plt.tight_layout()
    plt.savefig(OUT / "03_terminos_frecuentes.png", dpi=160, bbox_inches="tight")
    plt.show()

4. Comparación de experimentos y análisis de errores

In [ ]:
comparacion = RESULTS / "comparacion_experimentos_con_dummy.csv"
if not comparacion.exists():
    comparacion = RESULTS / "comparacion_experimentos.csv"
if comparacion.exists():
    exp = pd.read_csv(comparacion)
    display(exp)
    if {"experimento", "f1_macro"}.issubset(exp.columns):
        ax = exp.plot.bar(x="experimento", y="f1_macro", legend=False, figsize=(9, 4), rot=20)
        ax.set(title="F1 macro por experimento", ylabel="F1 macro", ylim=(0, 1), xlabel="")
        plt.tight_layout()
        plt.savefig(OUT / "04_comparacion_f1.png", dpi=160, bbox_inches="tight")
        plt.show()
else:
    print("No se encontró CSV de comparación")

pred_path = RESULTS / "predicciones_validacion.csv"
if pred_path.exists():
    pred = pd.read_csv(pred_path)
    print("Columnas de predicciones:", list(pred.columns))
    if {"experimento", "real", "prediccion"}.issubset(pred.columns):
        display(pd.crosstab([pred["experimento"], pred["real"]], pred["prediccion"]))
    else:
        print("Adaptar nombres de columnas para mostrar la matriz de confusión")
else:
    print("No se encontró CSV de predicciones")

In [ ]:
resumen = pd.DataFrame([{
    "total_expedientes": len(df),
    "dentro_alcance": int(df["dentro_alcance"].eq("SI").sum()),
    "fuera_alcance": int(df["dentro_alcance"].eq("NO").sum()),
    "pendientes": int(df["dentro_alcance"].eq("PENDIENTE").sum()),
    "aptos": int(df["apto_bool"].sum()),
    "aptos_admisible": int((df["apto_bool"] & df["etiqueta_inicial"].eq("ADMISIBLE")).sum()),
    "aptos_inadmisible": int((df["apto_bool"] & df["etiqueta_inicial"].eq("INADMISIBLE")).sum()),
    "aptos_improcedente": int((df["apto_bool"] & df["etiqueta_inicial"].eq("IMPROCEDENTE")).sum()),
}])
resumen.to_csv(OUT / "resumen_eda.csv", index=False, encoding="utf-8-sig")
display(resumen)
print("Gráficos y resumen guardados en:", OUT)